In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import expm
from scipy.optimize import minimize_scalar

DATA_FILE = Path('spallation_check.bin')

raw = np.fromfile(DATA_FILE, dtype=np.float64)

data = raw.reshape(-1, 5)
x, y, z, energy, species_raw = data.T

species = np.rint(species_raw).astype(int)


print(f'{len(data):,} records; {DATA_FILE.stat().st_size / 1e6:.2f} MB')
print('Species IDs found:', np.unique(species))

10,000 records; 0.40 MB
Species IDs found: [-1  0  1  2  3  4  5  6  7  8  9]


In [5]:
# Source-defined isotope order: IDs 0..9
isotopes = ['O16', 'N15', 'N14', 'C13', 'C12', 'B11', 'B10', 'Be10', 'Be9', 'Be7']
valid_ids = np.r_[np.arange(10), -1]
labels = isotopes + ['sink']
counts = pd.Series(species).value_counts().reindex(valid_ids, fill_value=0)
summary = pd.DataFrame({'count': counts.to_numpy(), 'fraction': counts.to_numpy() / len(species)}, index=labels)
display(summary)

,count,fraction
O16,631,0.0631
N15,332,0.0332
N14,239,0.0239
C13,244,0.0244
C12,493,0.0493
B11,539,0.0539
B10,279,0.0279
Be10,99,0.0099
Be9,165,0.0165
Be7,287,0.0287


In [13]:
sigma_1e27 = np.array([
    [293.2, 60.77, 30.07, 21.66, 33.16, 26.83, 10.40, 4.07, 3.35, 8.70],
    [0, 279.7, 20.15, 46.14, 43.82, 36.48, 13.59, 3.40, 7.98, 5.34],
    [0, 0, 265.9, 16.26, 52.00, 29.22, 10.16, 2.70, 2.71, 12.1],
    [0, 0, 0, 255.7, 56.73, 41.65, 4.70, 6.55, 7.52, 4.70],
    [0, 0, 0, 0, 237.1, 52.41, 18.08, 4.05, 7.32, 9.48],
    [0, 0, 0, 0, 0, 211.9, 40.85, 12.95, 15.15, 4.56],
    [0, 0, 0, 0, 0, 0, 197.1, 0, 13.95, 21.27],
    [0, 0, 0, 0, 0, 0, 0, 197.0, 36.37, 36.94],
    [0, 0, 0, 0, 0, 0, 0, 0, 181.7, 45.17],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 150.0],
]) * 1e-27
c_cm_s = 2.99792458e10
yr_s = 31557600.0
T_yr = 1e8
n_h = 0.1
abundance = np.zeros(11)
abundance[0] = 1e4
rate_factor = n_h * c_cm_s * yr_s

def predicted_fractions(n_h):
    # Column-vector convention: dp/dt = Q @ p
    Q = np.zeros((11, 11))
    for parent in range(10):
        rate_factor = n_h * c_cm_s * yr_s
        Q[parent, parent] -= rate_factor * sigma_1e27[parent, parent]
        for daughter in range(parent + 1, 10):
            Q[daughter, parent] += rate_factor * sigma_1e27[parent, daughter]
        sink_sigma = sigma_1e27[parent, parent] - sigma_1e27[parent, parent + 1:].sum()
        Q[10, parent] += rate_factor * sink_sigma
    return expm(Q * T_yr) @ abundance
final = predicted_fractions(n_h)
final

array([ 624.18961813,  382.78913712,  241.56886864,  264.69489074,
        483.90923972,  533.52599414,  292.05723828,  100.07256855,
        158.13224025,  265.48756474, 6653.57263969])

In [18]:
(np.array(counts) - final)/final

array([ 0.01091076, -0.13268176, -0.0106341 , -0.07818394,  0.01878609,
        0.01026005, -0.04470781, -0.01071791,  0.04343048,  0.08102992,
        0.00577545])